<a href="https://colab.research.google.com/github/loukikjoshi06-ops/NIRVANA-gpt/blob/main/NIRVANA_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F


here dataset is uploaded

In [2]:
from google.colab import files
uploaded = files.upload()

Saving TinyStories-small.txt to TinyStories-small.txt


In [3]:
with open('TinyStories-small.txt','r', encoding='utf-8')as l:
    text = l.read()

In [4]:
print(text[ :100])

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with


In [5]:
char = sorted(list(set(text)))
print (len(char))

106


In [6]:
stoi = {ch:i for i,ch in enumerate(char)}
itos = {i:ch for i,ch in enumerate(char)}


In [7]:
encode = lambda s:[stoi[c] for c in s]
decode = lambda l:''.join([itos[i] for i in l])

In [8]:
test = "loukik"
print(encode(test))
print(decode(encode(test)))

[71, 74, 80, 70, 68, 70]
loukik


In [9]:
data =torch.tensor(encode(text),dtype = torch.long)

In [10]:
print(data.shape)
print(data[:50])

torch.Size([52391770])
tensor([45, 73, 64,  2, 63, 60, 84, 12,  2, 60,  2, 71, 68, 79, 79, 71, 64,  2,
        66, 68, 77, 71,  2, 73, 60, 72, 64, 63,  2, 42, 68, 71, 84,  2, 65, 74,
        80, 73, 63,  2, 60,  2, 73, 64, 64, 63, 71, 64,  2, 68])


In [11]:
n = int(0.8 * len(data))
train_data = data[ :n]
val_data = data [n: ]

In [12]:
batch_size = 32
block_size = 64

In [13]:
def get_batch(split):
  data = train_data if split == 'train' else val_data

  # Ensure 'size' is a tuple by adding a comma: (batch_size,)
  ix = torch.randint(len(data) - block_size , (batch_size,))

  x = torch.stack([data[ i : i+block_size] for i in ix])
  y = torch.stack([data[ i+1 : i+block_size+1] for i in ix])

  return x,y

In [14]:
xb,yb = get_batch('train')
print(xb.shape)
print(yb.shape)

torch.Size([32, 64])
torch.Size([32, 64])


In [ ]:
for b in range(1):
  for t in range(block_size):
    context = xb[b,:t+1]
    target = yb[b,t]
    print(context.tolist(),"-->",target.item())

In [16]:
class BigramLanguageModel (nn.Module):
  def __init__(self):
   super().__init__()
   vocab_size = len(char) # Define vocab_size using the global 'char' variable
   self.token_embedding_table = nn.Embedding (vocab_size,vocab_size)

  def forward(self,idx,target = None):
    logits = self.token_embedding_table(idx)

    if target == None:
      loss = None
      return logits, loss # Explicitly return logits and None for loss

    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = target.view(B*T)

      loss = F.cross_entropy(logits,targets)
      return logits,loss

  def generate(self,idx,max_new_tokens):
    for _ in range(max_new_tokens):
      logits,loss = self(idx)

      logits = logits[:,-1,:]

      probs = F.softmax(logits,dim=-1)

      idx_next = torch.multinomial(probs,num_samples=1)

      idx = torch.cat((idx,idx_next),dim=1)

    return idx

In [17]:
model = BigramLanguageModel()

xb,yb = get_batch('train')

logits,loss = model(xb,yb)
print("Logit shape : ", logits.shape)
print("Loss : ", loss)

Logit shape :  torch.Size([2048, 106])
Loss :  tensor(5.0999, grad_fn=<NllLossBackward0>)


In [21]:
context = torch.zeros((1,1),dtype = torch.long)

generated = model.generate(context,max_new_tokens = 100)

print(decode(generated[0].tolist()))


	Tou
Teawed ire n clle cschofowim. s s ppaind me plus w!"Whe?” tea ho Ith frey is wino croogst awad, 


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
batch_size = 32

for iter in range(10000):

  xb,yb = get_batch('train')

  logits,loss = model(xb,yb)

  optimizer.zero_grad(set_to_none = True)
  loss.backward()
  optimizer.step()

  print(loss.item())

In [31]:
class Head(nn.Module):

  def __init__(self, head_size):
    super().__init__()

    self.key = nn.Linear(n_embd, head_size, bias = False)
    self.query = nn.Linear(n_embd, head_size, bias = False)
    self.value = nn.Linear(n_embd, head_size, bias = False)

    self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))

    self.dropout = nn.Dropout(dropout)


In [34]:
def forward(self,x):

  B,T,C = x.shape

  k = self.key(x)
  q = self.query(x)

  wei = q @ k.transpose(-2, -1)
  wei = wei * C** - 0.5
  wei = wei.masked_fill(self.tril[:T , :T] == 0, float('-inf'))
  wei = F.softmax(wei, dim = -1)
  wei = self.dropout(wei)
  v = self.value(x)
  out = wei @ v

  return out